# 05 — Feature Scaling

**Difficulty:** ⭐⭐⭐ &nbsp;&nbsp;|&nbsp;&nbsp; **Estimated Time:** 2.5 hours  
**Theme:** Credit Card Fraud Detection  
**Libraries:** numpy, pandas, matplotlib, seaborn, scikit-learn

---

## Learning Objectives
By the end of this notebook you will be able to:
1. Explain **why** feature scaling matters and which algorithms require it
2. Apply **MinMaxScaler**, **StandardScaler**, and **RobustScaler** correctly
3. Demonstrate the impact of outliers on each scaling method
4. Implement the **correct train-only fit** workflow to prevent data leakage
5. Empirically compare KNN accuracy with and without scaling

---
## 1. Why Does This Matter for ML?

Imagine you are building a fraud detection system. Your dataset has:

| Feature | Typical Range | Units |
|---|---|---|
| `amount` | \$0 – \$50,000 | dollars |
| `account_age_days` | 0 – 3,650 | days |
| `num_transactions` | 1 – 500 | count |
| `credit_score` | 300 – 850 | score |

Many ML algorithms (KNN, SVM, Neural Networks, Logistic Regression) compute **distances** or use **gradient descent**. When features are on wildly different scales, the algorithm pays far more attention to the feature with the largest numbers — not because it is more important, but simply because its values are bigger.

A \$50,000 transaction difference completely drowns out a 500-point difference in `num_transactions`. The model ends up effectively ignoring the smaller-scale features entirely.

**Feature scaling** puts every feature on a comparable numeric scale so the algorithm treats them fairly.

---
## 2. Real-World Analogy: The Olympic Judging Problem

Picture an Olympic gymnastics competition where two judges score routines:
- **Judge A** uses a **1–10** scale (artistic impression)
- **Judge B** uses a **0–100** scale (technical difficulty)

If you just add the scores together, Judge B's scores will **always dominate** the total — not because technical difficulty matters more, but because the numbers are 10× bigger. A perfect 10 from Judge A gets buried under an 85 from Judge B.

To combine them fairly, you need to **normalize** both judges onto the same scale first.

**This is exactly what feature scaling does for your ML model.** Your `amount` feature (Judge B) would otherwise shout over `num_transactions` (Judge A) in any distance-based or gradient-based algorithm.

---
## 3. Setup — Imports & Synthetic Dataset

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Scikit-learn tools ───────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ── Reproducibility ──────────────────────────────────────────────────────────
np.random.seed(42)

# ── Plot aesthetics ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('All imports successful.')

In [ ]:
# ── Generate synthetic credit-card transaction dataset ───────────────────────
# We create 2000 transactions: 1800 legitimate, 200 fraudulent.

n_legit  = 1800   # number of legitimate transactions
n_fraud  = 200    # number of fraudulent transactions
n_total  = n_legit + n_fraud

# --- Legitimate transactions ---
# Small-to-moderate amounts; older accounts; moderate transaction frequency
legit_amount       = np.random.exponential(scale=200, size=n_legit).clip(1, 5000)
legit_age_days     = np.random.randint(180, 3650, size=n_legit).astype(float)
legit_num_tx       = np.random.randint(5, 500, size=n_legit).astype(float)
legit_credit_score = np.random.normal(700, 50, size=n_legit).clip(300, 850)

# --- Fraudulent transactions ---
# Tend to have larger amounts, newer accounts, fewer previous transactions
fraud_amount       = np.random.exponential(scale=1500, size=n_fraud).clip(100, 50000)
fraud_age_days     = np.random.randint(0, 365, size=n_fraud).astype(float)
fraud_num_tx       = np.random.randint(1, 50, size=n_fraud).astype(float)
fraud_credit_score = np.random.normal(580, 80, size=n_fraud).clip(300, 850)

# --- Combine into a single DataFrame ---
df = pd.DataFrame({
    'amount'         : np.concatenate([legit_amount, fraud_amount]),
    'account_age_days': np.concatenate([legit_age_days, fraud_age_days]),
    'num_transactions': np.concatenate([legit_num_tx, fraud_num_tx]),
    'credit_score'   : np.concatenate([legit_credit_score, fraud_credit_score]),
    'is_fraud'       : np.array([0]*n_legit + [1]*n_fraud)
})

# Shuffle rows so fraud/legit are interleaved (not in blocks)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f"Fraud rate: {df['is_fraud'].mean():.1%}")
df.describe().round(2)

---
## 4. Visualising the Raw Distributions

Before scaling, look at the raw feature distributions. Notice how wildly different the x-axes are — `amount` stretches to 50,000 while `credit_score` sits in the 300–850 band. This is the problem we need to fix.

In [ ]:
# ── 4-panel histogram of raw features ────────────────────────────────────────
features = ['amount', 'account_age_days', 'num_transactions', 'credit_score']
colors   = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Raw Feature Distributions (before scaling)', fontsize=15, fontweight='bold', y=1.01)

for ax, feat, color in zip(axes.ravel(), features, colors):
    # Plot separate histograms for fraud vs legit to see overlap
    ax.hist(df.loc[df.is_fraud == 0, feat], bins=40, alpha=0.6,
            color='steelblue', label='Legitimate', density=True)
    ax.hist(df.loc[df.is_fraud == 1, feat], bins=40, alpha=0.6,
            color='tomato',    label='Fraud',      density=True)
    ax.set_title(feat, fontweight='bold')
    ax.set_ylabel('Density')
    # Show min/max range in the title subtitle
    ax.set_xlabel(f'Range: [{df[feat].min():.0f}, {df[feat].max():.0f}]')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('\nNotice: the x-axis scales are COMPLETELY different across features.')
print('This is why distance-based algorithms (KNN, SVM) will be biased toward large-scale features.')

---
## 5. Which Algorithms Need Scaling?

### Algorithms that ARE sensitive to scale (scaling required)

| Algorithm | Why? |
|---|---|
| **K-Nearest Neighbours (KNN)** | Uses Euclidean distance — large-scale features dominate |
| **Support Vector Machines (SVM)** | Maximises margin in feature space — unbalanced scales distort the margin |
| **Neural Networks** | Gradient descent converges much faster when features are in similar ranges |
| **Logistic Regression** (gradient descent) | Same reason as neural networks |
| **PCA** | Variance is scale-dependent — large-scale features will capture most of the first PC |
| **Ridge / Lasso Regression** | Regularisation term penalises coefficients equally — needs equal scales |

### Algorithms that are NOT sensitive to scale (scaling optional)

| Algorithm | Why? |
|---|---|
| **Decision Trees** | Splits on thresholds (e.g., `amount > 500`), not distances — scale irrelevant |
| **Random Forests** | Ensemble of decision trees — same reason |
| **XGBoost / LightGBM / CatBoost** | All gradient-boosted trees — threshold-based splits |
| **Naive Bayes** | Models probability distributions per feature independently |

> **Rule of thumb:** If the algorithm uses distances, dot products, or gradient descent — scale your features.

---
## 6. Min-Max Normalization — `MinMaxScaler`

**Plain English:** "Where does this value sit between the minimum and maximum?"

### Formula
$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

- Output range: **always [0, 1]** (or a custom [a, b] if you specify `feature_range`)
- The minimum value in training data maps to **0**; the maximum maps to **1**

### When to use
- Neural networks (inputs bounded in [0, 1] or [-1, 1])
- Image pixel values (divide by 255)
- When you know the absolute bounds of your data

### Weakness — outlier sensitivity
One extreme outlier (e.g., a \$500,000 fraudulent transaction) sets a new maximum. All the other values get squeezed into a tiny portion near 0. We will demonstrate this shortly.

In [ ]:
# ── Apply MinMaxScaler to the `amount` feature ────────────────────────────────
# We use reshape(-1, 1) because sklearn scalers expect a 2-D array.
# -1 means 'figure out the row count automatically'; 1 means 1 column.
amount_values = df['amount'].values.reshape(-1, 1)

# Instantiate the scaler
minmax_scaler = MinMaxScaler()

# fit_transform = fit() + transform() in one step
# fit()      → learns x_min and x_max from the data
# transform()→ applies the formula x_scaled = (x - min) / (max - min)
amount_minmax = minmax_scaler.fit_transform(amount_values)

print('=== MinMaxScaler on `amount` ===')
print(f'  Original  — min: {amount_values.min():>10.2f}, max: {amount_values.max():>10.2f},'
      f' mean: {amount_values.mean():>8.2f}')
print(f'  MinMax    — min: {amount_minmax.min():>10.4f}, max: {amount_minmax.max():>10.4f},'
      f' mean: {amount_minmax.mean():>8.4f}')

# The scaler stores what it learned: x_min and x_max
print(f'\n  Learned data_min_: {minmax_scaler.data_min_[0]:.2f}')
print(f'  Learned data_max_: {minmax_scaler.data_max_[0]:.2f}')

# Verify the formula manually for the first value
x0 = amount_values[0, 0]
manual_scaled = (x0 - minmax_scaler.data_min_[0]) / (minmax_scaler.data_max_[0] - minmax_scaler.data_min_[0])
print(f'\n  Manual check for row 0: ({x0:.2f} - {minmax_scaler.data_min_[0]:.2f}) / '
      f'({minmax_scaler.data_max_[0]:.2f} - {minmax_scaler.data_min_[0]:.2f}) = {manual_scaled:.6f}')
print(f'  sklearn output:         {amount_minmax[0, 0]:.6f}  ✓ matches!')

---
## 7. Z-Score Standardization — `StandardScaler`

**Plain English:** "How many standard deviations away from the average is this value?"

### Formula
$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

where $\mu$ is the mean and $\sigma$ is the standard deviation of the feature.

- Output: **mean = 0, std = 1** (unbounded — can be negative, can exceed ±3)
- A value at the average → 0; a value one std above average → +1

### When to use
- Most ML algorithms: SVM, Logistic Regression, PCA, Neural Networks
- When your data has outliers (they get mapped far out, but they don't distort all other values)
- When you are unsure which scaler to use — StandardScaler is the safest default

### Key difference from MinMax
A \$500,000 outlier will scale to ~20+ standard deviations — it stays extreme, but the other \$100–\$5000 transactions still spread nicely around 0. MinMax would have compressed all of them to near 0.

In [ ]:
# ── Apply StandardScaler to the `amount` feature ─────────────────────────────
standard_scaler = StandardScaler()

# fit_transform learns mean and std, then applies the z-score formula
amount_standard = standard_scaler.fit_transform(amount_values)

print('=== StandardScaler on `amount` ===')
print(f'  Original  — mean: {amount_values.mean():>8.2f}, std: {amount_values.std():>8.2f}')
print(f'  Standard  — mean: {amount_standard.mean():>8.4f}, std: {amount_standard.std():>8.4f}')
print(f'  Range after scaling: [{amount_standard.min():.2f}, {amount_standard.max():.2f}]')

# The scaler stores mean_ and scale_ (= std)
print(f'\n  Learned mean_:  {standard_scaler.mean_[0]:.4f}')
print(f'  Learned scale_: {standard_scaler.scale_[0]:.4f}')

# Verify the formula manually for the first value
x0 = amount_values[0, 0]
manual_z = (x0 - standard_scaler.mean_[0]) / standard_scaler.scale_[0]
print(f'\n  Manual z-score for row 0: ({x0:.2f} - {standard_scaler.mean_[0]:.2f}) / '
      f'{standard_scaler.scale_[0]:.2f} = {manual_z:.6f}')
print(f'  sklearn output:           {amount_standard[0, 0]:.6f}  ✓ matches!')

# Note: 99.7% of values in a normal distribution lie within ±3 std dev
# If a scaled value is outside ±3, it is likely an outlier
n_outliers = np.sum(np.abs(amount_standard) > 3)
print(f'\n  Values beyond ±3 std dev (potential outliers): {n_outliers}')

---
## 8. Robust Scaler — `RobustScaler`

**Plain English:** "Where does this value sit relative to the middle 50% of the data?"

### Formula
$$x_{\text{scaled}} = \frac{x - \text{median}}{\text{IQR}}$$

where IQR = Q3 − Q1 (the interquartile range — the span of the middle 50% of data).

### Why it is robust to outliers
- **Median** is not affected by extreme values (unlike mean)
- **IQR** only depends on the 25th–75th percentile (unlike std, which is pulled by outliers)
- A \$500,000 fraud transaction does not change the median of 2000 transactions

### When to use
- Data with **significant outliers you cannot remove** (e.g., real fraud amounts)
- When you want outliers to remain detectable but not distort the scale for other rows

### sklearn: `RobustScaler()`

In [ ]:
# ── Apply RobustScaler to the `amount` feature ────────────────────────────────
robust_scaler = RobustScaler()

# fit_transform learns the median and IQR, then applies the formula
amount_robust = robust_scaler.fit_transform(amount_values)

print('=== RobustScaler on `amount` ===')
print(f'  Original  — median: {np.median(amount_values):>8.2f},  '
      f'IQR: {np.percentile(amount_values, 75) - np.percentile(amount_values, 25):.2f}')

# The scaler stores center_ (median) and scale_ (IQR)
print(f'  Learned center_ (median): {robust_scaler.center_[0]:.4f}')
print(f'  Learned scale_  (IQR):   {robust_scaler.scale_[0]:.4f}')

print(f'\n  Scaled — mean: {amount_robust.mean():>6.4f}')
print(f'  Scaled — range: [{amount_robust.min():.2f}, {amount_robust.max():.2f}]')

# Verify manually
x0 = amount_values[0, 0]
manual_robust = (x0 - robust_scaler.center_[0]) / robust_scaler.scale_[0]
print(f'\n  Manual check for row 0: ({x0:.2f} - {robust_scaler.center_[0]:.2f}) / '
      f'{robust_scaler.scale_[0]:.2f} = {manual_robust:.6f}')
print(f'  sklearn output:         {amount_robust[0, 0]:.6f}  ✓ matches!')

---
## 9. Side-by-Side Comparison of All Three Scalers

Let us visually compare what each scaler does to the `amount` distribution.

In [ ]:
# ── Visualise the three scalers side by side ─────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Effect of Scalers on the `amount` Feature', fontsize=14, fontweight='bold')

# Panel 1: raw (unscaled)
axes[0].hist(amount_values, bins=50, color='#95a5a6', edgecolor='white')
axes[0].set_title('Original\n(unscaled)')
axes[0].set_xlabel('Amount ($)')

# Panel 2: MinMax scaled
axes[1].hist(amount_minmax, bins=50, color='#e74c3c', edgecolor='white')
axes[1].set_title('MinMaxScaler\n(range [0, 1])')
axes[1].set_xlabel('Scaled value')

# Panel 3: Standard scaled
axes[2].hist(amount_standard, bins=50, color='#3498db', edgecolor='white')
axes[2].set_title('StandardScaler\n(mean=0, std=1)')
axes[2].set_xlabel('Scaled value (z-score)')

# Panel 4: Robust scaled
axes[3].hist(amount_robust, bins=50, color='#27ae60', edgecolor='white')
axes[3].set_title('RobustScaler\n(median=0, IQR-based)')
axes[3].set_xlabel('Scaled value')

# Add range annotation to each panel
for ax, arr, label in zip(axes[1:], [amount_minmax, amount_standard, amount_robust],
                           ['MinMax', 'Standard', 'Robust']):
    # Print range inside the plot
    ax.text(0.97, 0.97,
            f'Range:\n[{arr.min():.2f}, {arr.max():.2f}]',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.7))

plt.tight_layout()
plt.show()

print('Observation: the SHAPE of the distribution does not change — only the x-axis scale.')
print('Scaling is a linear transformation; it cannot turn a skewed distribution into a normal one.')

---
## 10. Outlier Experiment — Injecting a \$500,000 Transaction

Fraud amounts can reach extreme values. Here we inject a single \$500,000 transaction and observe how each scaler reacts.

In [ ]:
# ── Outlier experiment ───────────────────────────────────────────────────────
# Take the original amount column and append ONE extreme outlier
outlier_value    = 500_000  # a single $500,000 fraudulent transaction
amounts_with_outlier = np.append(amount_values.ravel(), outlier_value).reshape(-1, 1)

print(f'Dataset size with outlier: {len(amounts_with_outlier):,} rows')
print(f'Outlier value: ${outlier_value:,.0f}')
print(f'Previously largest amount: ${amount_values.max():,.2f}')

# Apply all three scalers to the data WITH the outlier
minmax_with_out   = MinMaxScaler().fit_transform(amounts_with_outlier)
standard_with_out = StandardScaler().fit_transform(amounts_with_outlier)
robust_with_out   = RobustScaler().fit_transform(amounts_with_outlier)

# Now compare: what did a "typical" $1000 transaction scale to
# BEFORE and AFTER adding the outlier?
typical_amount = 1000.0

# Without outlier (use original scaler objects fitted earlier)
mm_before = minmax_scaler.transform([[typical_amount]])[0, 0]
st_before = standard_scaler.transform([[typical_amount]])[0, 0]
rb_before = robust_scaler.transform([[typical_amount]])[0, 0]

# With outlier (re-fit on the extended data)
mm_after  = MinMaxScaler().fit(amounts_with_outlier).transform([[typical_amount]])[0, 0]
st_after  = StandardScaler().fit(amounts_with_outlier).transform([[typical_amount]])[0, 0]
rb_after  = RobustScaler().fit(amounts_with_outlier).transform([[typical_amount]])[0, 0]

print(f'\nScaled value of a TYPICAL $1,000 transaction:')
print(f'  {'Scaler':<18} {'Without outlier':>18} {'With $500K outlier':>20} {'Change':>10}')
print('  ' + '-'*70)
for name, bf, af in [('MinMaxScaler', mm_before, mm_after),
                     ('StandardScaler', st_before, st_after),
                     ('RobustScaler', rb_before, rb_after)]:
    print(f'  {name:<18} {bf:>18.4f} {af:>20.4f} {af - bf:>+10.4f}')

In [ ]:
# ── Visualise outlier impact ──────────────────────────────────────────────────
# Plot the distribution of NON-outlier points after scaling with/without the outlier
# Focus on the first 2000 rows (no outlier row) for a fair visual comparison

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Impact of a $500K Outlier on Each Scaler\n'
             '(Showing distribution of the other 2000 transactions)',
             fontsize=13, fontweight='bold')

scaler_names  = ['MinMaxScaler', 'StandardScaler', 'RobustScaler']
without_out   = [amount_minmax,   amount_standard,   amount_robust]
# For "with outlier": take all rows except the last one (the injected outlier)
with_out      = [minmax_with_out[:-1], standard_with_out[:-1], robust_with_out[:-1]]
colors_scaler = ['#e74c3c', '#3498db', '#27ae60']

for ax, name, wo, wi, color in zip(axes, scaler_names, without_out, with_out, colors_scaler):
    ax.hist(wo,   bins=40, alpha=0.6, color='steelblue', label='Without outlier', density=True)
    ax.hist(wi,   bins=40, alpha=0.6, color=color,       label='With $500K outlier', density=True)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Scaled value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Key takeaway:')
print('  MinMaxScaler  → ALL values squish toward 0 (outlier dominates the max)')
print('  StandardScaler → Distribution shifts slightly but stays readable')
print('  RobustScaler   → Almost no change — median/IQR are outlier-resistant')

---
## 11. Empirical Proof — KNN Accuracy Without vs With Scaling

Let us now **prove empirically** that scaling matters for KNN. We will train the same KNN classifier on the same data, with and without feature scaling, and compare accuracy.

In [ ]:
# ── Prepare features and labels ───────────────────────────────────────────────
X = df[['amount', 'account_age_days', 'num_transactions', 'credit_score']].values
y = df['is_fraud'].values

# Split into train (80%) and test (20%) sets
# stratify=y ensures the fraud ratio is the same in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape[0]} samples  (fraud: {y_train.sum()})')
print(f'Test set:      {X_test.shape[0]} samples  (fraud: {y_test.sum()})')

# ── Experiment 1: KNN WITHOUT scaling ────────────────────────────────────────
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)
pred_unscaled = knn_unscaled.predict(X_test)
acc_unscaled  = accuracy_score(y_test, pred_unscaled)

# ── Experiment 2: KNN WITH StandardScaler ────────────────────────────────────
# CRITICAL: fit the scaler on TRAINING data only, then transform both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # learn from train, transform train
X_test_scaled  = scaler.transform(X_test)        # transform test using train statistics

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
pred_scaled = knn_scaled.predict(X_test_scaled)
acc_scaled  = accuracy_score(y_test, pred_scaled)

print(f'\n{"KNN Accuracy Comparison":=<50}')
print(f'  Without scaling:  {acc_unscaled:.4f} ({acc_unscaled:.1%})')
print(f'  With StandardScaler: {acc_scaled:.4f} ({acc_scaled:.1%})')
print(f'  Improvement:      +{acc_scaled - acc_unscaled:.4f} ({(acc_scaled-acc_unscaled)*100:+.2f} pp)')

In [ ]:
# ── Detailed classification report for both models ────────────────────────────
print('=== Without Scaling ===')
print(classification_report(y_test, pred_unscaled,
                             target_names=['Legitimate', 'Fraud']))

print('=== With StandardScaler ===')
print(classification_report(y_test, pred_scaled,
                             target_names=['Legitimate', 'Fraud']))

# ── Also compare Logistic Regression across all three scalers ─────────────────
results = {}
for name, sc in [('No scaling (raw)',  None),
                 ('MinMaxScaler',       MinMaxScaler()),
                 ('StandardScaler',     StandardScaler()),
                 ('RobustScaler',       RobustScaler())]:
    if sc is None:
        Xtr, Xte = X_train, X_test
    else:
        # Fit on train, transform both — never fit on test!
        Xtr = sc.fit_transform(X_train)
        Xte = sc.transform(X_test)
    # max_iter=1000 to ensure convergence for Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(Xtr, y_train)
    results[name] = accuracy_score(y_test, lr.predict(Xte))

print('\n=== Logistic Regression Accuracy by Scaler ===')
for name, acc in results.items():
    bar = '█' * int(acc * 50)
    print(f'  {name:<22}  {acc:.4f}  {bar}')

---
## 12. Critical Rule — Fit the Scaler on Training Data Only

This is one of the most common and subtle mistakes in ML pipelines.

### The Data Leakage Problem

If you `fit` (or `fit_transform`) the scaler on the **entire** dataset before splitting, the scaler learns statistics (mean, std, min, max) that include information from the test set. The model then "sees" the test set indirectly during training — this is called **data leakage**.

Leakage makes your validation metrics optimistically biased. Your model looks great on your test set, but fails in production where it has never seen those specific statistics.

### The correct workflow
```
1. Split data → X_train, X_test
2. scaler.fit(X_train)           ← learn statistics from TRAIN only
3. X_train_scaled = scaler.transform(X_train)
4. X_test_scaled  = scaler.transform(X_test)  ← apply TRAIN statistics to test
```

Think of it this way: in production, when a new transaction arrives, you will use the statistics from your training data to scale it — not statistics from future transactions you haven't seen yet.

In [ ]:
# ── Demonstrate the data leakage problem ─────────────────────────────────────
# WRONG WAY: fit scaler on ALL data before splitting
print('=== WRONG WAY (data leakage) ===')
bad_scaler = StandardScaler()
X_all_scaled = bad_scaler.fit_transform(X)   # <-- leaks test statistics into the scaler!

# Now split AFTER scaling — the test set has already influenced the scaler
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X_all_scaled, y, test_size=0.2, random_state=42, stratify=y
)
knn_bad = KNeighborsClassifier(n_neighbors=5)
knn_bad.fit(X_train_bad, y_train_bad)
acc_bad = accuracy_score(y_test_bad, knn_bad.predict(X_test_bad))
print(f'  KNN accuracy (LEAKY pipeline): {acc_bad:.4f}')

print()
print('=== RIGHT WAY (no leakage) ===')
# Split FIRST, then fit scaler on train only
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
good_scaler    = StandardScaler()
X_train_r_sc   = good_scaler.fit_transform(X_train_r)  # fit on train ONLY
X_test_r_sc    = good_scaler.transform(X_test_r)        # transform test using train stats

knn_good = KNeighborsClassifier(n_neighbors=5)
knn_good.fit(X_train_r_sc, y_train_r)
acc_good = accuracy_score(y_test_r, knn_good.predict(X_test_r_sc))
print(f'  KNN accuracy (clean pipeline):  {acc_good:.4f}')

print()
print('Note: The leaky pipeline can sometimes appear better (optimistic bias),')
print('but it will NOT generalise to truly new data in production.')

In [ ]:
# ── Better practice: use sklearn Pipeline to make leakage impossible ──────────
# A Pipeline chains preprocessing and modelling steps.
# When you call pipeline.fit(X_train), it:
#   1. Fits the scaler on X_train
#   2. Transforms X_train with the fitted scaler
#   3. Trains the model on the scaled X_train
# When you call pipeline.predict(X_test), it:
#   1. Transforms X_test using the already-fitted scaler (no re-fitting)
#   2. Makes predictions
# This makes data leakage structurally impossible.

from sklearn.pipeline import Pipeline

# Build the pipeline: scaling → KNN
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),          # step name: 'scaler'
    ('knn',    KNeighborsClassifier(n_neighbors=5))  # step name: 'knn'
])

# Fit on training data only — pipeline handles the rest correctly
pipeline.fit(X_train, y_train)
acc_pipeline = accuracy_score(y_test, pipeline.predict(X_test))

print(f'Pipeline accuracy: {acc_pipeline:.4f}')
print('Using Pipeline is the recommended way to prevent data leakage automatically.')
print('\nPipeline steps:')
for step_name, step_obj in pipeline.steps:
    print(f'  {step_name}: {step_obj.__class__.__name__}')

---
## 13. Summary — Choosing the Right Scaler

| Scaler | Formula | Output Range | Best For | Weakness |
|---|---|---|---|---|
| **MinMaxScaler** | (x − min) / (max − min) | [0, 1] | Neural networks, bounded inputs | Extremely sensitive to outliers |
| **StandardScaler** | (x − μ) / σ | unbounded (μ=0, σ=1) | Most ML algorithms; the safe default | Slightly affected by outliers (mean/std are sensitive) |
| **RobustScaler** | (x − median) / IQR | unbounded | Data with known outliers you want to keep | Less common; output not bounded |

### Decision Checklist
1. **Tree-based model?** → No scaling needed at all
2. **Neural network?** → MinMaxScaler (if data is clean) or StandardScaler
3. **Data with significant outliers?** → RobustScaler
4. **Everything else (SVM, KNN, LogReg, PCA)?** → StandardScaler
5. **Always:** fit on training data only, transform both train and test

---
## 14. Self-Check Questions

Test your understanding. Try to answer each question before revealing the answer.

---

**Question 1:** You are building a neural network to detect fraud. The `amount` feature ranges from \$1 to \$50,000. Should you use MinMaxScaler or StandardScaler? What if there are occasional \$500,000 fraud attempts in production?

<details>
<summary>▶ Reveal Answer</summary>

**For clean data (no extreme outliers):** Use **MinMaxScaler** — neural networks work well with inputs in [0, 1] or [-1, 1], and it guarantees bounded inputs.

**If $500K fraud amounts exist:** MinMaxScaler becomes dangerous. That single outlier would set the new maximum, squishing all \$1–\$50K transactions into the range [0, 0.1]. The neural network would see essentially no variation in normal transaction amounts.

**Better solution for outlier-prone data:**  
- Use **RobustScaler** (resistant to outliers) or  
- Cap/clip the `amount` at the 99th percentile before applying MinMaxScaler, or  
- Use **StandardScaler** which handles outliers more gracefully (the outlier appears far out, but doesn't distort the bulk of the distribution).
</details>

---

**Question 2:** You fit MinMaxScaler on training data where `amount` has a maximum of \$50,000. At inference time, a transaction amount of \$75,000 arrives — higher than the training maximum. What is the scaled value, and is this a problem?

<details>
<summary>▶ Reveal Answer</summary>

The scaled value will be **greater than 1.0**:
```
scaled = (75,000 - training_min) / (50,000 - training_min)  > 1.0
```

**Is this a problem?** Yes and no:
- The formula still works mechanically — sklearn will not throw an error
- But the output violates the [0, 1] contract that MinMaxScaler promises
- For neural networks expecting inputs in [0, 1], a value of 1.5 can push activations into unexpected territories

**Solutions:**
1. Use **StandardScaler** instead — no hard boundary, so out-of-range values are just higher z-scores
2. Set `clip=True` in MinMaxScaler: `MinMaxScaler(clip=True)` — this clamps any value above the training max to 1.0
3. **Monitor for distribution shift:** a value that far outside the training range is a signal that something may have changed in your data
</details>

---

**Question 3:** Decision Tree classifiers do not need feature scaling. Why not? Is this also true for Random Forests and XGBoost?

<details>
<summary>▶ Reveal Answer</summary>

**Decision Trees split on thresholds, not distances.** At each node, the tree asks: "Is `amount` > \$500?" or "Is `credit_score` > 650?" It finds the threshold that best separates classes by checking every possible split point.

Because splits are based on **relative ordering** (which side of a threshold a value falls on), multiplying all `amount` values by 0.00002 (MinMax) does not change the ordering. The tree finds the same optimal threshold regardless.

**Yes, this applies to all tree-based ensembles:**
- **Random Forests** = many decision trees → no scaling needed
- **XGBoost / LightGBM / CatBoost** = gradient-boosted trees → no scaling needed
- These are among the most popular algorithms precisely because they handle heterogeneous feature scales naturally.

**However:** scaling never hurts trees — it just doesn't help them either. It is safe (but unnecessary) to scale before fitting a tree.
</details>

---

**Question 4:** You accidentally fit StandardScaler on your entire dataset before the train/test split. How does this cause data leakage, and why does it matter?

<details>
<summary>▶ Reveal Answer</summary>

**What happens:**  
When StandardScaler computes `mean` and `std`, it uses ALL rows — including the test set rows. Those test-set statistics then flow into every scaled value, even in the training data.

**Why this is leakage:**  
Your model indirectly "knows" information about the test set (its mean, its spread) before it has officially seen it. This is like a student knowing some of the exam answers while studying.

**Why it matters:**  
1. **Optimistic accuracy:** Your test set metric will be slightly too high — the model appears to generalise better than it truly does
2. **Production failure:** In production, new batches of data will be scaled using training statistics only. The scaling will be different from what the model was trained/validated with, causing a subtle but real performance degradation
3. **With small datasets:** The effect can be large — test statistics can significantly shift the mean/std compared to training-only statistics

**The correct workflow every time:**
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, ...)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # learn from train only
X_test_sc  = scaler.transform(X_test)        # apply train stats to test
```
</details>

---
## 15. Quick Reference Card

```python
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline

# ── Always: split first ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ── Option A: manual (correct order matters!) ──────────────────────────────
scaler = StandardScaler()                   # or MinMaxScaler() / RobustScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit + transform on train
X_test_sc  = scaler.transform(X_test)       # transform only on test

# ── Option B: Pipeline (leakage-proof, recommended) ───────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  KNeighborsClassifier(n_neighbors=5))
])
pipeline.fit(X_train, y_train)              # scaler fits on X_train internally
pipeline.predict(X_test)                    # uses train statistics automatically

# ── Choosing a scaler ──────────────────────────────────────────────────────
# Tree-based (RF, XGBoost)?  → No scaler needed
# Neural network, clean data → MinMaxScaler()
# Default / SVM / LogReg     → StandardScaler()
# Data with heavy outliers   → RobustScaler()
```